In [57]:
import pandas as pd

In [58]:
orders_df = pd.read_csv('../include/data/nammamart/orders.csv')
customers_df = pd.read_csv('../include/data/nammamart/customers.csv')
payments_df = pd.read_csv('../include/data/nammamart/payments.csv')
products_df = pd.read_csv('../include/data/nammamart/products.csv')
stores_df = pd.read_csv('../include/data/nammamart/stores.csv')
orders_incremental_df = pd.read_csv('../include/data/nammamart/orders_2026-10-01_increment.csv')

### Data sanity check for orders.csv

In [59]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          2412 non-null   object 
 1   order_ts          2410 non-null   object 
 2   customer_id       2394 non-null   object 
 3   store_id          2412 non-null   object 
 4   product_id        2412 non-null   object 
 5   quantity          2412 non-null   int64  
 6   unit_price        2412 non-null   float64
 7   discount_pct      2412 non-null   int64  
 8   order_amount      2412 non-null   float64
 9   payment_mode      2412 non-null   object 
 10  order_status      2412 non-null   object 
 11  channel           2412 non-null   object 
 12  delivery_minutes  2248 non-null   float64
dtypes: float64(3), int64(2), object(8)
memory usage: 245.1+ KB


In [60]:
orders_ts_in_dt = pd.to_datetime(orders_df['order_ts'], format="mixed", errors='coerce') 
order_amount_series = (orders_df['quantity'] * orders_df['unit_price'] * (1 - orders_df['discount_pct'] / 100)).round(2)
discrepancies = orders_df[abs(orders_df['order_amount'] - order_amount_series) > 0.01]
print(f"Total number of records in orders CSV are: {len(orders_df)}")
print(f"Are order IDs in order CSV unique: {orders_df['order_id'].is_unique}")
print(f"order IDs in orders CSV that are null: {orders_df['order_id'].isnull().sum()}")
print(f"Customer IDs in orders CSV that are null: {orders_df['customer_id'].isnull().sum()}")
print(f"Store IDs in orders CSV that are null: {orders_df['store_id'].isnull().sum()}")
print(f"Product IDs in orders CSV that are null: {orders_df['product_id'].isnull().sum()}")
print(f"Quantity in orders CSV that are null: {orders_df['quantity'].isnull().sum()}")
print(f"Quantity in orders CSV that is zero or negative: {len(orders_df[orders_df['quantity'] < 1])}")
print(f"Unit Price in orders CSV that are null: {orders_df['unit_price'].isnull().sum()}")
print(f"Unit Price in orders CSV that is zero or negative: {len(orders_df[orders_df['unit_price'] < 1])}")
print(f"Different payment modes present in orders CSV : {orders_df['payment_mode'].unique()}")
print(f"Different order status present in orders CSV : {orders_df['order_status'].unique()}")
print(f"Non-cancelled orders that have delivery time that is null: {orders_df[orders_df['order_status'] != 'CANCELLED']['delivery_minutes'].isnull().sum()}")
print(f"Different orders that have delivery time above 10 minutes KPI: {len(orders_df[orders_df['delivery_minutes'] > 10])}")
print(f"Different orders that have delivery time above 30 minutes KPI: {len(orders_df[orders_df['delivery_minutes'] > 30])}")
print(f"Different orders that have delivery time above 60 minutes KPI: {len(orders_df[orders_df['delivery_minutes'] > 60])}")
print(f"Orders where discount percentage is more than 100% : {len(orders_df[orders_df['discount_pct'] > 100])}")
print(f"Orders where discount percentage is negative : {len(orders_df[orders_df['discount_pct'] < 0])}")
print(f"Minimum order timestamp in Orders is: {orders_ts_in_dt.min()}")
print(f"Maximum order timestamp in Orders is: {orders_ts_in_dt.max()}")
print(f"Orders with null timestamp: {orders_df['order_ts'].isnull().sum()}")
print(f"Order amount is null for {order_amount_series.isnull().sum()}")
print(f"Order amount where the total order amount is zero or less: {len(order_amount_series[order_amount_series <= 0])}")
print(f"Order amount where there is descripency from the calculated one: {len(discrepancies)}")

Total number of records in orders CSV are: 2412
Are order IDs in order CSV unique: False
order IDs in orders CSV that are null: 0
Customer IDs in orders CSV that are null: 18
Store IDs in orders CSV that are null: 0
Product IDs in orders CSV that are null: 0
Quantity in orders CSV that are null: 0
Quantity in orders CSV that is zero or negative: 10
Unit Price in orders CSV that are null: 0
Unit Price in orders CSV that is zero or negative: 0
Different payment modes present in orders CSV : ['UPI' 'Card' 'Cash on Delivery' 'Wallet' 'Cashh' 'upi' 'CARD' 'UPI '
 'cod']
Different order status present in orders CSV : ['DELIVERED' 'RETURNED' 'CANCELLED' 'delivered' 'Pending??']
Non-cancelled orders that have delivery time that is null: 8
Different orders that have delivery time above 10 minutes KPI: 1992
Different orders that have delivery time above 30 minutes KPI: 418
Different orders that have delivery time above 60 minutes KPI: 6
Orders where discount percentage is more than 100% : 3
Orde

### Data sanity check for Payments.csv

In [61]:
payments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2228 entries, 0 to 2227
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   payment_id      2228 non-null   object 
 1   order_id        2228 non-null   object 
 2   payment_ts      2228 non-null   object 
 3   payment_mode    2228 non-null   object 
 4   amount_paid     2228 non-null   float64
 5   payment_status  2228 non-null   object 
 6   gateway_ref     1927 non-null   object 
dtypes: float64(1), object(6)
memory usage: 122.0+ KB


In [62]:
payments_ts_in_dt = pd.to_datetime(payments_df['payment_ts'], format="mixed", errors='coerce')
clean_payment_mode = payments_df['payment_mode'].str.strip().str.lower()
print(f"Total number of records in payments are: {len(payments_df)}")
print(f"Are payment IDs in payments CSV unique: {payments_df['payment_id'].is_unique}")
print(f"Number of duplicate payment IDs: {payments_df['payment_id'].duplicated().sum()}")
print(f"Number of duplicate order IDs in payments: {payments_df['order_id'].duplicated().sum()}")
print(f"Payment IDs that are null: {payments_df['payment_id'].isnull().sum()}")
print(f"Order IDs that are null: {payments_df['order_id'].isnull().sum()}")
print(f"Amounts paid that are null: {payments_df['amount_paid'].isnull().sum()}")
print(f"Payments that are zero or less for non-refunded orders: {len(payments_df[(payments_df['amount_paid'] <= 0) & (payments_df['payment_status'] != 'REFUNDED')])}")
print(f"Different payment statuses recorded: {payments_df['payment_status'].unique()}")
print(f"Raw payment modes present: {payments_df['payment_mode'].unique()}")
print(f"Standardized payment modes: {clean_payment_mode.unique()}")
print(f"Gateway refs that are null: {payments_df['gateway_ref'].isnull().sum()}")
print(f"Gateway refs null for non-cash modes (UPI, Card, Wallet): {payments_df[(clean_payment_mode.isin(['upi', 'card', 'wallet']))]['gateway_ref'].isnull().sum()}")
print(f"Minimum payment timestamp: {payments_ts_in_dt.min()}")
print(f"Maximum payment timestamp: {payments_ts_in_dt.max()}")
print(f"Payments with null timestamp: {payments_ts_in_dt.isnull().sum()}")

Total number of records in payments are: 2228
Are payment IDs in payments CSV unique: False
Number of duplicate payment IDs: 4
Number of duplicate order IDs in payments: 4
Payment IDs that are null: 0
Order IDs that are null: 0
Amounts paid that are null: 0
Payments that are zero or less for non-refunded orders: 10
Different payment statuses recorded: ['SUCCESS' 'FAILED' 'REFUNDED']
Raw payment modes present: ['UPI' 'Wallet' 'Cash on Delivery' 'Card' 'upi' 'cod' 'CARD' 'UPI '
 'Cashh']
Standardized payment modes: ['upi' 'wallet' 'cash on delivery' 'card' 'cod' 'cashh']
Gateway refs that are null: 301
Gateway refs null for non-cash modes (UPI, Card, Wallet): 0
Minimum payment timestamp: 2026-09-01 06:30:24
Maximum payment timestamp: 2027-01-01 09:01:46
Payments with null timestamp: 0


### Data sanity check for Customers.csv

In [63]:
customers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 405 entries, 0 to 404
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   customer_id   405 non-null    object
 1   full_name     405 non-null    object
 2   email         401 non-null    object
 3   phone         405 non-null    int64 
 4   city          399 non-null    object
 5   area          405 non-null    object
 6   pincode       405 non-null    object
 7   signup_date   405 non-null    object
 8   loyalty_tier  405 non-null    object
 9   is_active     405 non-null    object
dtypes: int64(1), object(9)
memory usage: 31.8+ KB


In [64]:
cust_signup_dt = pd.to_datetime(customers_df['signup_date'], format="mixed", errors='coerce')
phone_digits = customers_df['phone'].astype(str).str.strip()
phone_invalid_len = len(customers_df[~phone_digits.str.match(r'^\d{10}$')])
email_invalid_fmt = len(customers_df[~customers_df['email'].astype(str).str.contains(r'^[^@]+@[^@]+\.[^@]+$', regex=True, na=True)])

print(f"Total number of records in customers CSV: {len(customers_df)}")
print(f"Are customer IDs unique: {customers_df['customer_id'].is_unique}")
print(f"Number of duplicate customer IDs: {customers_df['customer_id'].duplicated().sum()}")
print(f"Customer IDs that are null: {customers_df['customer_id'].isnull().sum()}")
print(f"Full names that are null: {customers_df['full_name'].isnull().sum()}")
print(f"Emails that are null: {customers_df['email'].isnull().sum()}")
print(f"Phones that are null: {customers_df['phone'].isnull().sum()}")
print(f"Phone numbers that are NOT 10 digits: {phone_invalid_len}")
print(f"Emails with invalid format: {email_invalid_fmt}")
print(f"Cities present: {customers_df['city'].unique()}")
print(f"Pincodes that are null: {customers_df['pincode'].isnull().sum()}")
print(f"Different loyalty tiers present: {customers_df['loyalty_tier'].unique()}")
print(f"Different is_active values: {customers_df['is_active'].unique()}")
print(f"Minimum signup date: {cust_signup_dt.min()}")
print(f"Maximum signup date: {cust_signup_dt.max()}")
print(f"Signup dates that are null/unparseable: {cust_signup_dt.isnull().sum()}")

Total number of records in customers CSV: 405
Are customer IDs unique: False
Number of duplicate customer IDs: 5
Customer IDs that are null: 0
Full names that are null: 0
Emails that are null: 4
Phones that are null: 0
Phone numbers that are NOT 10 digits: 7
Emails with invalid format: 12
Cities present: ['Bengaluru' nan]
Pincodes that are null: 0
Different loyalty tiers present: ['SILVER' 'Silver' 'Platinum' 'Bronze' 'Gold' 'BRONZE' 'bronze']
Different is_active values: ['Y' 'N']
Minimum signup date: 2024-06-01 00:00:00
Maximum signup date: 2027-01-05 00:00:00
Signup dates that are null/unparseable: 0


### Data sanity check for Products.csv

In [65]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     152 non-null    object 
 1   product_name   152 non-null    object 
 2   category       149 non-null    object 
 3   sub_category   152 non-null    object 
 4   brand          152 non-null    object 
 5   mrp            152 non-null    float64
 6   cost_price     152 non-null    float64
 7   unit           152 non-null    object 
 8   is_perishable  152 non-null    bool   
 9   launch_date    152 non-null    object 
dtypes: bool(1), float64(2), object(7)
memory usage: 11.0+ KB


In [66]:
prod_launch_dt = pd.to_datetime(products_df['launch_date'], format="mixed", errors='coerce')
cost_gt_mrp = products_df[products_df['cost_price'] > products_df['mrp']]

print(f"Total number of records in products CSV: {len(products_df)}")
print(f"Are product IDs unique: {products_df['product_id'].is_unique}")
print(f"Number of duplicate product IDs: {products_df['product_id'].duplicated().sum()}")
print(f"Product IDs that are null: {products_df['product_id'].isnull().sum()}")
print(f"Product names that are null: {products_df['product_name'].isnull().sum()}")
print(f"Categories that are null: {products_df['category'].isnull().sum()}")
print(f"Different categories present: {products_df['category'].unique()}")
print(f"Sub-categories that are null: {products_df['sub_category'].isnull().sum()}")
print(f"Brands that are null: {products_df['brand'].isnull().sum()}")
print(f"MRP that is null: {products_df['mrp'].isnull().sum()}")
print(f"MRP that is zero or negative: {len(products_df[products_df['mrp'] <= 0])}")
print(f"Cost price that is null: {products_df['cost_price'].isnull().sum()}")
print(f"Cost price that is zero or negative: {len(products_df[products_df['cost_price'] <= 0])}")
print(f"Products where cost_price > mrp: {len(cost_gt_mrp)}")
print(f"Different units present: {products_df['unit'].unique()}")
print(f"Different is_perishable values: {products_df['is_perishable'].unique()}")
print(f"Minimum launch date: {prod_launch_dt.min()}")
print(f"Maximum launch date: {prod_launch_dt.max()}")

Total number of records in products CSV: 152
Are product IDs unique: False
Number of duplicate product IDs: 2
Product IDs that are null: 0
Product names that are null: 0
Categories that are null: 3
Different categories present: ['Fruits & Vegetables' 'Dairy & Bakery' nan 'Staples' 'Snacks & Beverages'
 'Personal Care' 'Household']
Sub-categories that are null: 0
Brands that are null: 0
MRP that is null: 0
MRP that is zero or negative: 1
Cost price that is null: 0
Cost price that is zero or negative: 0
Products where cost_price > mrp: 5
Different units present: ['kg' 'pack' 'unit']
Different is_perishable values: [ True False]
Minimum launch date: 2023-01-03 00:00:00
Maximum launch date: 2025-09-20 00:00:00


### Data sanity check for Stores.csv

In [67]:
stores_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   store_id                 15 non-null     object
 1   store_name               15 non-null     object
 2   zone                     14 non-null     object
 3   area                     15 non-null     object
 4   pincode                  15 non-null     int64 
 5   open_date                15 non-null     object
 6   manager_name             15 non-null     object
 7   capacity_orders_per_day  15 non-null     int64 
dtypes: int64(2), object(6)
memory usage: 1.1+ KB


In [68]:
print(f"Total number of records in stores CSV: {len(stores_df)}")
print(f"Are store IDs unique: {stores_df['store_id'].is_unique}")
print(f"Number of duplicate store IDs: {stores_df['store_id'].duplicated().sum()}")
print(f"Store IDs that are null: {stores_df['store_id'].isnull().sum()}")
print(f"Store names that are null: {stores_df['store_name'].isnull().sum()}")
print(f"Different zones present: {stores_df['zone'].unique()}")
print(f"Areas that are null: {stores_df['area'].isnull().sum()}")
print(f"Pincodes that are null: {stores_df['pincode'].isnull().sum()}")
print(f"Manager names that are null: {stores_df['manager_name'].isnull().sum()}")
print(f"Capacity that is null: {stores_df['capacity_orders_per_day'].isnull().sum()}")
print(f"Capacity that is zero or negative: {len(stores_df[stores_df['capacity_orders_per_day'] <= 0])}")
print(f"Minimum capacity per day: {stores_df['capacity_orders_per_day'].min()}")
print(f"Maximum capacity per day: {stores_df['capacity_orders_per_day'].max()}")

Total number of records in stores CSV: 15
Are store IDs unique: True
Number of duplicate store IDs: 0
Store IDs that are null: 0
Store names that are null: 0
Different zones present: ['North' 'South' nan 'East' 'West']
Areas that are null: 0
Pincodes that are null: 0
Manager names that are null: 0
Capacity that is null: 0
Capacity that is zero or negative: 1
Minimum capacity per day: 0
Maximum capacity per day: 600


### Data sanity check for orders_2026-10-01_increment.csv

In [69]:
orders_incremental_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          160 non-null    object 
 1   order_ts          160 non-null    object 
 2   customer_id       160 non-null    object 
 3   store_id          160 non-null    object 
 4   product_id        160 non-null    object 
 5   quantity          160 non-null    int64  
 6   unit_price        160 non-null    float64
 7   discount_pct      160 non-null    int64  
 8   order_amount      160 non-null    float64
 9   payment_mode      160 non-null    object 
 10  order_status      160 non-null    object 
 11  channel           160 non-null    object 
 12  delivery_minutes  160 non-null    int64  
dtypes: float64(2), int64(3), object(8)
memory usage: 16.4+ KB


In [70]:
inc_ts_in_dt = pd.to_datetime(orders_incremental_df['order_ts'], format="mixed", errors='coerce')
overlap_mask = orders_incremental_df['order_id'].isin(orders_df['order_id'])
overlap_orders = orders_incremental_df[overlap_mask]
new_orders = orders_incremental_df[~overlap_mask]

print(f"Total number of records in incremental orders CSV: {len(orders_incremental_df)}")
print(f"Are order IDs in incremental CSV unique: {orders_incremental_df['order_id'].is_unique}")
print(f"Number of duplicate order IDs: {orders_incremental_df['order_id'].duplicated().sum()}")
print(f"Number of orders updating existing September orders: {len(overlap_orders)}")
print(f"Status breakdown of updated orders:\n{overlap_orders['order_status'].value_counts()}")
print(f"Number of new orders for October 1: {len(new_orders)}")
print(f"Minimum order timestamp in Incremental: {inc_ts_in_dt.min()}")
print(f"Maximum order timestamp in Incremental: {inc_ts_in_dt.max()}")
print(f"Different order statuses in Incremental: {orders_incremental_df['order_status'].unique()}")
print(f"Different payment modes in Incremental: {orders_incremental_df['payment_mode'].unique()}")

Total number of records in incremental orders CSV: 160
Are order IDs in incremental CSV unique: True
Number of duplicate order IDs: 0
Number of orders updating existing September orders: 10
Status breakdown of updated orders:
order_status
RETURNED    10
Name: count, dtype: int64
Number of new orders for October 1: 150
Minimum order timestamp in Incremental: 2026-09-01 09:38:13
Maximum order timestamp in Incremental: 2026-10-01 22:33:00
Different order statuses in Incremental: ['RETURNED' 'DELIVERED']
Different payment modes in Incremental: ['UPI' 'Cash on Delivery' 'Wallet' 'Card']


### Referencial Integrity

In [71]:
# Check orphaned customer IDs in orders
orphaned_customers = orders_df[~orders_df['customer_id'].isin(customers_df['customer_id']) & orders_df['customer_id'].notnull()]

# Check orphaned product IDs in orders
orphaned_products = orders_df[~orders_df['product_id'].isin(products_df['product_id']) & orders_df['product_id'].notnull()]

# Check orphaned store IDs in orders
orphaned_stores = orders_df[~orders_df['store_id'].isin(stores_df['store_id']) & orders_df['store_id'].notnull()]

# Check orphaned order IDs in payments
orphaned_payments = payments_df[~payments_df['order_id'].isin(orders_df['order_id']) & payments_df['order_id'].notnull()]

# Check orders without any payment record
unpaid_orders = orders_df[~orders_df['order_id'].isin(payments_df['order_id'])]

print(f"Orders with non-existent customer_id: {len(orphaned_customers)}")
print(f"Orders with non-existent product_id: {len(orphaned_products)}")
print(f"Orders with non-existent store_id: {len(orphaned_stores)}")
print(f"Payments with non-existent order_id: {len(orphaned_payments)}")
print(f"Orders without payment records: {len(unpaid_orders)} (Breakdown by status: {unpaid_orders['order_status'].value_counts().to_dict()})")


Orders with non-existent customer_id: 0
Orders with non-existent product_id: 9
Orders with non-existent store_id: 5
Payments with non-existent order_id: 8
Orders without payment records: 184 (Breakdown by status: {'CANCELLED': 156, 'DELIVERED': 27, 'RETURNED': 1})
